# Reversible transformers on a 6 GB laptop GPU

**ERA V5, Session 13 assignment.** Train a ~20M-parameter LLM for 50M tokens at a fixed batch size, train it again
with a reversible layer stack (and report which variant works: midpoint, Euler, ...), then train the reversible
model again at the largest batch that fits. Report final loss, speed (tokens/s), peak memory and other findings.

Everything runs locally: the data is downloaded into `data/`, every run writes its log to `results/<run>.log` and
its numbers to `results/<run>.json`, and this notebook keeps all training logs in its cell outputs.

| | |
|---|---|
| model | GPT, 10 layers, width 384, 6 heads, context 512, learned positions, tied embeddings, **21.05M parameters** |
| data | TinyStories (one 249 MB training shard, 117M tokens; validation split 4.7M tokens) |
| tokenizer | byte-level BPE, vocab 8,192, trained here on 200k stories |
| optimizer | AdamW (0.9, 0.95), peak lr 1e-3, 2% warmup, cosine to 10%, grad clip 1.0 |
| regularization | dropout 0 and weight decay 0 in every run (reversibility needs a deterministic forward pass) |
| precision | fp32 (fp16 matmuls run 9x slower than fp32 on this GPU, measured below) |
| loss | cross-entropy computed in chunks of 2,048 tokens in every run, so the 8,192-wide logits never dominate memory |

In [ ]:
import json, os, platform, time
import numpy as np, torch
import matplotlib.pyplot as plt
import revlm
from revlm import GPT, probe, max_batch, run_or_load

print("python", platform.python_version(), "| torch", torch.__version__, "| CUDA", torch.version.cuda)
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name}, {p.total_memory / 2**30:.2f} GiB, {p.multi_processor_count} SMs, compute {p.major}.{p.minor}")
print("GPU state now:", revlm.gpu_state())
# WDDM would otherwise spill into shared system RAM instead of failing; cap the allocator at free dedicated VRAM
print(f"PyTorch allocator capped at {revlm.limit_to_dedicated_vram():,.0f} MiB of dedicated VRAM")

## 1. Data

`prep_data.py` downloads TinyStories from the Hugging Face hub, trains the tokenizer and writes `data/train.bin`
and `data/val.bin` as `uint16` token ids. It skips the work when the files already exist.

In [ ]:
import prep_data
prep_data.main()
from tokenizers import Tokenizer
tok = Tokenizer.from_file("data/tokenizer.json")
val = np.memmap("data/val.bin", dtype=np.uint16, mode="r")
print("\nsample of the validation stream:\n", tok.decode(val[:120].tolist()))

## 2. Why fp32 on this card

The GTX 1660 Ti (Turing TU116) has no tensor cores. A 4096 x 4096 matrix multiply:

In [ ]:
for dt in (torch.float32, torch.float16):
    a = torch.randn(4096, 4096, device="cuda", dtype=dt); b = torch.randn_like(a)
    for _ in range(3): a @ b
    torch.cuda.synchronize(); t = time.time()
    for _ in range(20): a @ b
    torch.cuda.synchronize()
    print(f"{str(dt):14s} {20 * 2 * 4096**3 / (time.time() - t) / 1e12:.2f} TFLOPS")
del a, b; torch.cuda.empty_cache()

## 3. The reversible stack

Write $p^\ell$ for the residual stream entering layer $\ell$ and $f_\ell(p) = \mathrm{Attn}_\ell(\mathrm{LN}_1 p) + \mathrm{MLP}_\ell(\mathrm{LN}_2(p + \mathrm{Attn}_\ell(\mathrm{LN}_1 p)))$
for the whole block (Gal et al. 2025, *Reversing Large Language Models for Efficient Training and Fine-Tuning*, Eq. 5).
`revlm.py` implements four reversible rules:

| variant | forward | inverse (used in the backward pass) |
|---|---|---|
| **midpoint** (Eq. 4) | $p^{\ell+1} = p^{\ell-1} + 2h\,f_\ell(p^\ell)$ | $p^{\ell-1} = p^{\ell+1} - 2h\,f_\ell(p^\ell)$ |
| **Euler** (symplectic / Hamiltonian, Eqs. 8-9, $a=b=1$) | $q^{\ell+1} = q^\ell + h\,\mathrm{Attn}(\mathrm{LN}_1 p^\ell)$; $p^{\ell+1} = p^\ell + h\,\mathrm{MLP}(\mathrm{LN}_2 q^{\ell+1})$ | $p^\ell = p^{\ell+1} - h\,\mathrm{MLP}(\cdot)$; $q^\ell = q^{\ell+1} - h\,\mathrm{Attn}(\cdot)$ |
| **leapfrog** (Eq. 6) | $p^{\ell+1} = 2p^\ell - p^{\ell-1} + h^2 f_\ell(p^\ell)$ | $p^{\ell-1} = 2p^\ell - p^{\ell+1} + h^2 f_\ell(p^\ell)$ |
| **blend**, "midpoint (a)" (Eq. 15, fixed $a$) | $p^{\ell+1} = a\,p^{\ell-1} + (1-a)\,p^\ell + h\,f_\ell(p^\ell)$ | $p^{\ell-1} = (p^{\ell+1} - (1-a)p^\ell - h f_\ell(p^\ell))/a$ |

Both streams start at the embedding ($p^{-1} = p^0$, or $q^0 = p^0$), and the head reads the last state through a final
LayerNorm. The whole stack is one `torch.autograd.Function`: the forward pass runs under `no_grad` and keeps only the
final pair of states; the backward pass walks the layers in reverse, rebuilds each layer's input from its output with
the inverse rule, and runs that layer once more with autograd on to get its gradients. Each block is therefore
evaluated twice per step (forward, and once in the backward pass), against once for the standard stack.

**Correctness check.** In float64 the reversible backward pass must give the same gradients as ordinary autograd
applied to the same update rule:

In [ ]:
import check_grads
check_grads.main()

## 4. Memory and speed probes

A few training steps per configuration, measuring `torch.cuda.max_memory_allocated` and step time.
`checkpoint` is ordinary activation recomputation (stores each block's input, re-runs the block in the backward pass),
included as the alternative the session notes compare against.

**Caveat on speed:** this laptop GPU runs at 85-92°C and throttles (nvidia-smi reports thermal slowdown and drops the
memory clock from 6000 to 810 MHz for stretches), so short probes are noisy. The full runs below average over hours.

In [ ]:
rows = []
for v in ["standard", "checkpoint", "euler", "midpoint"]:
    for b in [8, 16, 32]:
        r = probe(v, b, steps=6, h=0.5 if v == "midpoint" else 1.0)
        rows.append((v, b, r))
        print(f"{v:10s} batch {b:3d}: " + (f"peak {r['peak_mib']:7,.0f} MiB  {r['tok_s']:8,.0f} tok/s" if r else "OOM"), flush=True)

**Memory against depth.** The reversible stack's activation memory should not grow with the number of layers.
Batch 8 x 512 tokens, width 384, depth 10 to 80. The training state (fp32 weight, gradient and two Adam moments,
16 bytes per parameter) grows with depth in every variant, so the split below separates it from the rest:

In [ ]:
depth = {}
for v in ["standard", "checkpoint", "euler"]:
    for L in [10, 20, 40, 80]:
        r = probe(v, 8, steps=2, n_layer=L)
        depth[(v, L)] = r
        print(f"{v:10s} {L:3d} layers: " + (f"peak {r['peak_mib']:7,.0f} MiB = training state {r['state_mib']:6,.0f} "
              f"+ activations/workspace {r['act_mib']:6,.0f} MiB" if r else "OOM (over the dedicated-VRAM cap)"), flush=True)

**Largest batch that fits** (doubling, then a binary search; batch = sequences of 512 tokens). Other programs on the
laptop hold some GPU memory, so these are the limits on this machine as it is used.

In [ ]:
os.makedirs("results", exist_ok=True)
if os.path.exists("results/max_batch.json"):  # keep the first measurement when the notebook is resumed
    mb = json.load(open("results/max_batch.json"))
    print("(loaded from results/max_batch.json, measured in an earlier execution)")
else:
    mb = {}
    for v in ["standard", "checkpoint", "euler", "midpoint"]:
        t = time.time()
        mb[v] = max_batch(v, start=16, h=0.5 if v == "midpoint" else 1.0)
        print(f"{v:10s} probed in {time.time() - t:.0f}s", flush=True)
    json.dump(mb, open("results/max_batch.json", "w"), indent=1)
for v, b in mb.items():
    print(f"{v:10s} max batch {b:4d}  ({b * 512:,} tokens/step)")

## 5. Run 1: standard transformer, fixed batch 16 x 512 = 8,192 tokens/step

6,103 steps for 50M tokens. Batch 16 fits the standard model with room to spare (batch 32 already uses ~4.4 GB of the
6 GB card, and the desktop holds close to 1 GB).

In [ ]:
FIXED = dict(batch=16, tokens=50_000_000, log_every=100, eval_every=500)
res = {}
res["standard_b16"] = run_or_load(name="standard_b16", variant="standard", **FIXED)

## 6. Run 2: reversible, same batch 16, Euler and midpoint

Euler with $h = 1$ (each half-step adds a full attention or MLP update, like a standard residual block); midpoint with
$h = 0.5$ (so $2h = 1$ and each update has the same scale).

In [ ]:
res["euler_b16"] = run_or_load(name="euler_b16", variant="euler", h=1.0, **FIXED)

In [ ]:
res["midpoint_b16"] = run_or_load(name="midpoint_b16", variant="midpoint", h=0.5, **FIXED)

**Short screens of the other two rules** (5M tokens each, same batch, same learning-rate shape compressed to 5M
tokens): leapfrog with $h = 1$, and the blend rule with the Lightning LM settings, $h = 0.25$, $a = 0.5$.

In [ ]:
SCREEN = dict(batch=16, tokens=5_000_000, log_every=50, eval_every=200)
res["leapfrog_b16_5M"] = run_or_load(name="leapfrog_b16_5M", variant="leapfrog", h=1.0, **SCREEN)

In [ ]:
res["blend_b16_5M"] = run_or_load(name="blend_b16_5M", variant="blend", h=0.25, alpha=0.5, **SCREEN)

In [ ]:
best = min(["euler", "midpoint"], key=lambda v: res[f"{v}_b16"]["final_val_loss"])
for v in ["euler", "midpoint"]:
    r = res[f"{v}_b16"]
    print(f"{v:9s} final val loss {r['final_val_loss']:.4f}  train {r['final_train_loss']:.4f}  {r['tok_per_s']:,.0f} tok/s  "
          f"peak {r['peak_alloc_mib']:,.0f} MiB  reconstruction rel err {r['reconstruction_rel_err']:.1e}")
print("-> variant used for the max-batch run:", best)

## 7. Run 3: reversible at the maximum batch size

The batch is the probed maximum for the chosen variant, less 10% so that allocator fragmentation over a long run
cannot push it out of memory. The token budget stays at 50M, so a larger batch means fewer optimizer steps; the peak
learning rate is scaled by $\sqrt{B/16}$ (capped at 3e-3) and warmup is 5% of the steps.

In [ ]:
B = int(mb[best] * 0.9)
lr = min(3e-3, 1e-3 * (B / 16) ** 0.5)
steps = 50_000_000 // (B * 512)
print(f"{best}: probed max batch {mb[best]}, training at batch {B} ({B * 512:,} tokens/step, {steps} steps), lr {lr:.2e}")
res["rev_maxbatch"] = run_or_load(name="rev_maxbatch", variant=best, h=1.0 if best == "euler" else 0.5, batch=B,
                                  lr=lr, warmup_frac=0.05, tokens=50_000_000,
                                  log_every=max(5, steps // 60), eval_every=max(10, steps // 12) * 1)

## 8. Results

In [ ]:
import pandas as pd
order = ["standard_b16", "euler_b16", "midpoint_b16", "rev_maxbatch", "leapfrog_b16_5M", "blend_b16_5M"]
tab = pd.DataFrame([{
    "run": k, "variant": res[k]["cfg"]["variant"], "batch": res[k]["cfg"]["batch"],
    "tokens (M)": round(res[k]["tokens_seen"] / 1e6, 1), "steps": res[k]["steps"],
    "final train loss": round(res[k]["final_train_loss"], 4), "final val loss": round(res[k]["final_val_loss"], 4),
    "tokens/s": round(res[k]["tok_per_s"]), "peak alloc (MiB)": round(res[k]["peak_alloc_mib"]),
    "peak reserved (MiB)": round(res[k]["peak_reserved_mib"]), "train time (min)": round(res[k]["train_minutes"], 1),
    "reconstruction rel err": (f"{res[k]['reconstruction_rel_err']:.1e}" if "reconstruction_rel_err" in res[k] else "-"),
} for k in order if k in res]).set_index("run")
tab.to_csv("results/summary.csv")
tab

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for k in ["standard_b16", "euler_b16", "midpoint_b16", "rev_maxbatch"]:
    h, c = res[k]["hist"], res[k]["cfg"]
    mtok = np.array(h["step"]) * c["batch"] * c["ctx"] / 1e6
    ax[0].plot(mtok, h["loss"], label=k)
    ax[1].plot(np.array(h["val_step"]) * c["batch"] * c["ctx"] / 1e6, h["val_loss"], "o-", ms=3, label=k)
    ax[2].plot(mtok, np.array(h["tok_s"]) / 1e3, label=k, lw=0.8)
ax[0].set(title="train loss", xlabel="tokens (M)", ylim=(1.0, 4.0))
ax[1].set(title="validation loss", xlabel="tokens (M)", ylim=(1.0, 4.0))
ax[2].set(title="throughput (thermal throttling shows as dips)", xlabel="tokens (M)", ylabel="k tokens/s")
for a in ax: a.grid(alpha=0.3); a.legend()
plt.tight_layout(); plt.savefig("results/loss_curves.png", dpi=120); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for v in ["standard", "checkpoint", "euler", "midpoint"]:
    pts = [(b, r["peak_mib"]) for vv, b, r in rows if vv == v and r]
    ax[0].plot(*zip(*pts), "o-", label=v)
ax[0].set(title="peak memory vs batch (10 layers)", xlabel="batch (x512 tokens)", ylabel="MiB")
for v in ["standard", "checkpoint", "euler"]:
    pts = [(L, r["act_mib"]) for (vv, L), r in depth.items() if vv == v and r]
    ax[1].plot(*zip(*pts), "o-", label=v)
ax[1].set(title="peak minus training state vs depth (batch 8)", xlabel="layers", ylabel="MiB")
ks = ["standard_b16", "euler_b16", "midpoint_b16", "rev_maxbatch"]
ax[2].bar(ks, [res[k]["peak_alloc_mib"] for k in ks], color=["C0", "C1", "C2", "C3"])
ax[2].set(title="peak memory of the full runs", ylabel="MiB")
ax[2].tick_params(axis="x", rotation=15)
for a in ax[:2]: a.grid(alpha=0.3); a.legend()
plt.tight_layout(); plt.savefig("results/memory.png", dpi=120); plt.show()

In [ ]:
# Text samples from the final weights of each 50M-token run (checkpoints/ is kept out of git)
for k in ["standard_b16", "euler_b16", "midpoint_b16", "rev_maxbatch"]:
    c = res[k]["cfg"]
    m = GPT(variant=c["variant"], h=c["h"], alpha=c["alpha"]).cuda()
    m.load_state_dict(torch.load(f"checkpoints/{k}.pt"))
    print("---", k)
    print(revlm.generate(m, tok, "Once upon a time, a little dog named Max"))
    print()
    del m

## 9. Findings

_(written after the runs finish)_